# Tutorial 2: an agent that drives the simulator

In the first tutorial **you** built a model, ran it and looked at the results. Here a program does those steps for you: you
give it a request in plain English, and it decides which `kusim` commands to run, reads the answers, fixes mistakes and
reports. The "brain" is a large language model (LLM) such as Mistral or Claude; everything else is ordinary Python that you can read.

By the end you will have:

1. built the **loop** (observe, act, check) that makes a program an *agent*, and run it on three tasks;
2. connected it to **any model** by changing three settings (or to a scripted model that needs no key);
3. asked it to **draw plots** in plain words, and read what it did in the trace.

**What you need.** The kusim package and the Python packages of tutorial 1. For a live model, a free API key (section 2). Without a key
the notebook runs a *scripted* model that makes the same tool calls a careful model makes, so you can do everything in this
notebook offline and get the same result every time.

**Cost and privacy.** Free plans have small quotas and some providers may use your prompts to train their models; everything here is
synthetic data, so nothing private leaves your computer. Never paste real company data into a free service.

**How an agent works.** It is a loop that repeats until the model gives an answer:

1. You give a task. The program sends the model the task, the list of tools and everything said so far. The model only ever sees text.
2. The model replies with either a final answer or a **tool call**, for example "call `run` with `model=model.json`".
3. The program checks the call, asks **you** first if it is expensive, runs `kusim run ...` and reads its JSON answer.
4. The answer, shortened, is added to the conversation, and the loop goes back to step 2.

The model never touches your files. It can only ask; the program decides what is allowed.

## 0. Set up

The helper modules are in this folder: `agent_lab.py` (the connection, the tools, the loop), `agent_lab_case.py` (the files the tasks work on) and `agent_lab_scripts.py` (the three requests and the scripted plans).

In [ ]:
import importlib.util
import json
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd() if Path("agent_lab.py").exists() else Path("examples")      # this notebook's folder
sys.path.insert(0, str(HERE.resolve()))
import agent_lab as al
import agent_lab_case as case
import agent_lab_scripts as sc

for module in (al, case, sc):              # pick up a newer version of the helper files without restarting the kernel
    importlib.reload(module)

import kusim
print("kusim", kusim.__version__)

In [ ]:
# a fresh folder for the agent to work in (the agent can not leave it), and the "hidden truth" next to it
shutil.rmtree(HERE / "output" / "agent_lab", ignore_errors=True)
folders = case.make_workspace(HERE / "output" / "agent_lab")
WORK = folders["work"]
for path in sorted(WORK.rglob("*")):
    if path.is_file():
        print(path.relative_to(WORK).as_posix())

`model.json` is for Task A, `model_b.json` for Task B and the folder `field/` for Task C: a deck, measured data with the field's well
names, and the description of what is uncertain. The real permeability that produced those data is kept in `hidden/`; the agent
has no tool that can read it, and the agent has no way to see it.

## 1. The tools: the command line, described to a model

A model can only use what it is told about. kusim was built so that every capability is a command with documented options and
one JSON answer, so the tool descriptions can be **generated** from the command line itself (`kusim.cli.build_parser()`). Add a command or
an option to kusim and the agent gets it.

In [ ]:
tools, commands = al.kusim_tools()
rows = []
for t in tools:
    if t["name"] in commands:                                    # a kusim command: show the command and its options
        command, layout = commands[t["name"]]
        options = " ".join(flag or f"<{key}>" for key, flag, _, _ in layout)
        rows.append((t["name"], f"kusim {command}", options, t["description"][:95]))
    else:                                                        # list_files, read_file, write_file: Python helpers of the agent
        rows.append((t["name"], "(local helper, not a kusim command)", "", t["description"][:95]))
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows, columns=["tool", "command it runs", "options", "what the model is told"])

In [ ]:
# what the model receives for one tool: a name, a description and a JSON Schema of the arguments
edit = next(t for t in tools if t["name"] == "edit")
print(json.dumps(edit, indent=1)[:1500])

Three things are *not* in the list on purpose: the guardrail limits (`max_cells`, `max_seconds`, ... an agent may lower them, not
raise them), editing a model in place (an agent writes a new file), and a way out of the workspace folder. `ensemble` has caps
(at most 4 workers and 100 members here) and **waits for your approval**, because it runs many simulations.

## 2. Connect a model

All providers below speak the same "chat with tools" protocol, except Anthropic, which has its own; `agent_lab.connect()` hides the
difference. Choose with three environment variables, **set before you start Jupyter** (or in a cell, see below):

| Provider | `KUSIM_LLM_PROVIDER` | Key variable | Where to get a key |
|---|---|---|---|
| **Mistral** (the default) | `mistral` | `MISTRAL_API_KEY` | console.mistral.ai, the free "Experiment" plan |
| Groq | `groq` | `GROQ_API_KEY` | console.groq.com, free plan (limits are per model) |
| Anthropic | `anthropic` | `ANTHROPIC_API_KEY` | console.anthropic.com |
| OpenRouter | `openrouter` | `OPENROUTER_API_KEY` | openrouter.ai (some models are free) |
| Ollama (on your computer) | `ollama` | none | ollama.com; pull a model that supports tools |
| anything OpenAI-compatible | `custom` | `KUSIM_LLM_API_KEY` | also set `KUSIM_LLM_BASE_URL` and `KUSIM_LLM_MODEL` |
| no model | `scripted` | none | replays a plan |

`KUSIM_LLM_MODEL` picks another model of the provider (the defaults in `agent_lab.PRESETS` were right when written; providers retire
models and change free quotas often, so check the provider's page). In PowerShell: `$env:MISTRAL_API_KEY = "..."` then start `jupyter lab`.

In [ ]:
# Optional: type a key for this session only (it is not saved in the notebook). Remove the '#' to use it.
# import getpass; os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral key: ")

In [ ]:
try:
    al.connect()                      # reads KUSIM_LLM_PROVIDER (default: mistral) and the key
    LIVE = True
    print("live model:", al.connect().model, "at", al.connect().base_url)
except al.LLMError as problem:
    LIVE = False
    print(problem)
    print("\n-> continuing with the scripted model: the same tool calls, no key needed.")


def new_agent(task, **kwargs):
    """An agent for 'A', 'B' or 'C': the live model if there is a key, else the scripted plan of that task."""
    llm = al.connect() if LIVE else al.connect("scripted", script=sc.SCRIPTS[task])
    (WORK / f"trace_{task}.jsonl").unlink(missing_ok=True)
    return al.Agent(llm, WORK, approve=al.ask_user if LIVE else al.auto_approve, trace=f"trace_{task}.jsonl", **kwargs)

**Cannot connect? Check the network first.** If the cells above or below fail with a message about a web page, a reset connection, a certificate or a timeout, the cause is usually not your key but the network: a company firewall or web filter that blocks AI services, or one that inspects HTTPS traffic. `al.diagnose` asks the provider's server (without your key) and tells you whether it can be reached from this computer, and if not, what it saw instead (for example the block page text) and what to do about it: ask IT to allow the address, set `HTTPS_PROXY`, set `SSL_CERT_FILE` to the company certificate, try another network, or use the local notebook (Ollama) instead.

In [ ]:
al.diagnose()

**Say hello first.** Before any tools: can you talk to the model? The cell below sends a greeting and prints the answer. This only checks the connection (key, address, model name) and shows the reply, the time and the token count. Change `GREETING`, or call `talk("...")` again to keep the conversation going: the model remembers what was said before in `history`.
A model does not always know its own name, so trust the "live model" line above, not what the model says about itself.

In [ ]:
import time

GREETING = "Hello! Who are you, and what can you help me with in a reservoir-simulation course?"     # change it, or call talk("...") yourself

llm = al.connect() if LIVE else None

history = [{"role": "system", "content": "You are a friendly assistant in a reservoir-simulation course. Answer in two or three sentences."}]


def talk(text):
    """Send one message, with the conversation so far, and print the model's answer."""
    history.append({"role": "user", "content": text})
    started = time.time()
    reply = llm.chat(history, [])                                   # no tools here: just a conversation
    history.append({"role": "assistant", "content": reply.text or "(no answer)"})
    print(f"you  : {text}\nmodel: {reply.text.strip() or '(an empty answer: try again, or check the model name)'}")
    print(f"({time.time() - started:.1f} s, tokens: {reply.usage})")


if LIVE:
    talk(GREETING)
else:
    print("(skipped: no live model; the scripted model does not talk)")

In [ ]:
# one test call: does the model ask for a tool when we ask it to? (only with a live model)
if LIVE:
    llm = al.connect()
    answer = llm.chat([{"role": "user", "content": "Call the tool list_files once, with no arguments."}],
                      [t for t in tools if t["name"] == "list_files"])
    print("the model asked for:", [(c.name, c.arguments) for c in answer.tool_calls] or answer.text)
else:
    print("(skipped: no live model)")

## 3. You are the agent first

Before the loop, do what the loop does, by hand. `helper.call(tool, **arguments)` makes one tool call with the same checks the agent's calls get,
and returns the answer as a dict. Answers are long, so the agent sees a shortened form (`al.compact`): long lists cut, floats
rounded, the keys that matter (`ok`, `error`, `code`, `hint`) first.

In [ ]:
helper = al.Agent(al.connect("scripted"), WORK, verbose=False)
answer = helper.call("validate", model="model.json")
print("validate -> runnable:", answer["runnable"], "| findings:", answer["checks"]["findings"])

In [ ]:
# a mistake on purpose: an unknown option is refused BEFORE anything runs, and the answer says what is allowed
print(al.compact(helper.call("edit", model="model.json", rate=300, output="x.json"), 400))
# and a wrong path inside the model: kusim's own error is located and has hints
print(al.compact(helper.call("edit", model="model.json", set=["producers.0.tarjet=2000"], output="x.json"), 500))

In [ ]:
helper.call("edit", model="model.json", set=["producers.0.target=2300"], output="by_hand.json")
answer = helper.call("run", model="by_hand.json", out="runs/by_hand")
p1 = answer["wells"]["P1"]
print(f"healthy: {answer['healthy']} | P1 after {answer['final_time']:g} days: oil rate {p1['oil_rate']:.1f} STB/d, cumulative oil {p1['cum_oil']:,.0f} STB")

Now the same thing as a **loop**. This is the whole idea in about 15 lines (the real `Agent.run` adds the checks, the approval, the trace and the step limit):

In [ ]:
def tiny_agent(llm, task, helper, max_steps=8):
    messages = [{"role": "system", "content": al.SYSTEM_PROMPT}, {"role": "user", "content": task}]
    for _ in range(max_steps):
        reply = llm.chat(messages, helper.tools)                      # 1. the model looks and decides
        if not reply.tool_calls:
            return reply.text                                          #    no tool wanted: it answers
        messages.append({"role": "assistant", "content": reply.text or None, "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.name, "arguments": json.dumps(c.arguments)}} for c in reply.tool_calls]})
        for call in reply.tool_calls:
            result = helper.call(call.name, **call.arguments)         # 2. the program acts
            messages.append({"role": "tool", "tool_call_id": call.id, "content": al.compact(result)})   # 3. and tells the model
    return "(step limit)"


plan = [{"tool_calls": [{"name": "validate", "arguments": {"model": "by_hand.json"}}]},
        lambda messages: {"text": "validate said runnable = " + str(sc.results(messages)[-1]["runnable"])}]
print(tiny_agent(al.ScriptedLLM(plan), "Is by_hand.json fine to run?", helper))

## 4. Task A: a what-if

> *The model is model.json. What happens to the cumulative oil of producer P1 at the end of the run if the producer's bottom-hole pressure is lowered from 2500 to 2000 psi? Keep model.json as it is; run both cases and compare.*

**Predict first.** A lower bottom-hole pressure means more drawdown. Will cumulative oil go up or down, and by roughly how much (a few
percent or a factor of two)? Write your guess down, then run the agent.

In [ ]:
agent = new_agent("A")
run = agent.run(sc.TASKS["A"])

Read the **trace**: one line per tool call. A good agent validates before it runs, writes a new file instead of overwriting, and says what it did *not* check.

In [ ]:
def show_trace(run):
    rows = []
    for event in al.read_trace(run.trace):
        if event["event"] == "tool":
            args = ", ".join(f"{k}={v}" for k, v in event["arguments"].items())
            rows.append((event["step"], event["tool"], args[:90], event["ok"], (json.loads(event["result"]).get("code") or "") if not event["ok"] else ""))
    return pd.DataFrame(rows, columns=["step", "tool", "arguments", "ok", "error code"])


show_trace(run)

**Questions.** Was your prediction right? Which of its calls would you have skipped, and which
would you have added? (An agent that does not call `validate` is not wrong here, but it would be on Task B.)

## 5. Task B: fix the run

> *Run model_b.json and tell me the final oil rate and water cut of P1. I want numbers I can trust: if anything is wrong with the run or the model, fix it in a copy and say what you changed.*

`model_b.json` has a time step of 25 days and a high injection rate. IMPES (the explicit saturation update) is only conditionally stable, so
the run will **finish** but be wrong. Watch how the agent finds out, and how it reacts when one of its own calls is refused.

In [ ]:
agent = new_agent("B")
run = agent.run(sc.TASKS["B"])
show_trace(run)

Notice why this works: a run can end with `"ok": true` and still be wrong, so kusim reports `healthy` and `diagnostics`, `validate` gives a
`suggested_dt`, and the agent was told (in `SYSTEM_PROMPT`) to check them. Errors that say *what* is wrong and *how to fix it*
are what let a program recover without a human.

## 6. Task C: from field files to a conclusion

> *The folder field/ has what the field team sent: the deck waterflood.DATA, history.csv (measured data, day 0 is the deck's START, 1 January 2020) and later.csv (data after the history). Import the deck and the history, check how well the deck model reproduces the history, calibrate the permeability with ES-MDA (parameters.json and prior.json are in field/), and report. Use the later data to check the calibrated model. Write your conclusion in report.md.*

First look at what the agent is given: an export in the field's names and units (bar, m3/d, percent), with blank cells.

In [ ]:
print((WORK / "field" / "waterflood.DATA").read_text().splitlines()[0])
pd.read_csv(WORK / "field" / "history.csv").head(6)

The agent will import the deck (`import_deck`), then the data (`import_history`: units, well names, sigma, 30-day intervals), run
`misfit`, then **`ensemble`**, which asks for your **approval** (type `y` with a live model; the scripted run approves automatically and says so).
The ensemble runs 40 simulations in 4 parallel processes, about 30 seconds.

In [ ]:
agent = new_agent("C", max_steps=15)
run = agent.run(sc.TASKS["C"])

In [ ]:
show_trace(run)

In [ ]:
print((WORK / "report.md").read_text())

## 7. Ask for a plot

The agent can draw what you ask for, with no code from you and none from the model. It never writes Python: it looks at the examples with `plot_template`,
at the columns of your files with `tables`, writes a **description of the figure** (a small JSON file: which table, which columns, which kind of trace) and
calls `plot`. kusim checks the description (only known trace types, no links, no maps, a size limit) and draws it. The pictures go to a file whose name the
agent chooses, so the cell below looks for the new pictures and shows them.

- **PNG files** need the Chrome or Edge browser on this computer (`pip install kusim[plot]` installs the drawing library). Without a browser ask for an `.html` page instead: it is interactive and needs nothing.
- **`keep=True`** continues the conversation of the last run, so the agent still knows the files and the models of Task C: "the best model" means the calibrated one it just made.
- The plot needs a **live model** (the scripted model only knows the three tasks); run Task C first.


In [ ]:
from IPython.display import IFrame, Image, display


def pictures():
    return {p: p.stat().st_mtime for ext in ("png", "html") for p in WORK.rglob(f"*.{ext}")}


def show_new(before):
    """The agent chooses the file names, so show every picture or page written (or rewritten) since `before`."""
    for path, mtime in sorted(pictures().items()):
        if before.get(path) != mtime:
            print(path.relative_to(WORK).as_posix())
            display(Image(filename=str(path)) if path.suffix == ".png" else IFrame(str(path.relative_to(Path.cwd())) if path.is_relative_to(Path.cwd()) else str(path), 900, 600))


In [ ]:
if LIVE:
    before = pictures()
    run = agent.run("Run the calibrated model, then plot the observed history against its simulated water cut for the producers. Write a png.", keep=True)
    show_new(before)
else:
    print("(skipped: plotting needs a live model)")


In [ ]:
if LIVE:
    before = pictures()
    run = agent.run("Now a map of the field with the wells labelled, coloured by how badly each well is matched.", keep=True)
    show_new(before)


Read the trace (`show_trace(run)`) to see how it worked: `plot_template`, `tables`, a new file written, `plot`, and a look at the `warnings` of the answer (a trace
with no rows draws nothing, and the model cannot see the picture, so it has to read those warnings).

**Try your own.** Ask in plain words, with `agent.run("...", keep=True)` (and `agent.forget()` to start a new conversation):

- the water saturation averaged over all layers, with the wells labelled, at the last time;
- the permeability of the top layer as a map;
- the average water saturation of every layer against depth;
- the observed and simulated bottom-hole pressure of one producer, for the deck model and the calibrated model in one figure;
- the mismatch of every well and quantity as bars.

3-D views and animations are not covered by this route (`examples/03_visualization.py` does them from Python).
